# Baselines determinístico e preditivo — M3

Este notebook demonstra a política fixa de melhor canal histórico, o DummyClassifier e a Regressão Logística de propensão. O treino usa somente `train`, o limiar é escolhido em `validation` e `test` é usado apenas na avaliação final.

O modelo estima associação `P(conversão | contexto, canal observado)`. Nenhum resultado deste notebook representa efeito causal do canal.

In [ ]:
from pathlib import Path
import json
import sys

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Localiza a raiz do projeto sem depender do diretório inicial do kernel.
current_path = Path.cwd().resolve()
project_root = current_path if (current_path / 'configs').exists() else current_path.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.models.train_propensity import (
    PROPENSITY_INPUT_COLUMNS,
    load_modeling_config,
    load_processed_splits,
    run_m3_training,
)

modeling_config_path = project_root / 'configs' / 'modeling.yaml'
sns.set_theme(style='whitegrid', palette='deep')

## 1. Execução reproduzível

A execução abaixo regenera os relatórios sem abrir um novo run no MLflow. O run oficial é criado pelo comando de linha de comando documentado no README.

In [ ]:
m3_report = run_m3_training(modeling_config_path, enable_tracking=False)
modeling_config = load_modeling_config(modeling_config_path)
splits, preparation_metadata = load_processed_splits(modeling_config)

pd.Series({
    'marco': m3_report['milestone'],
    'modelo_selecionado': m3_report['selection']['selected_model'],
    'gate_superou_dummy': m3_report['selection']['gate_passed'],
    'features_transformadas': m3_report['contracts']['transformed_feature_count'],
})

## 2. Baseline determinístico

In [ ]:
fixed_statistics = pd.DataFrame(m3_report['fixed_policy']['statistics'])
fixed_statistics

In [ ]:
replay_summary = pd.DataFrame([
    {'split': split_name, **metrics}
    for split_name, metrics in m3_report['fixed_policy']['replay'].items()
])[['split', 'recommended_action', 'accepted_events', 'replay_coverage', 'conversions', 'mean_reward']]
replay_summary

`celular` é congelado como melhor braço usando somente o treino. No replay, a recompensa existe apenas quando a recomendação coincide com o canal histórico; por isso, taxa de recompensa e cobertura devem ser apresentadas juntas.

## 3. Dummy versus Regressão Logística

In [ ]:
comparison_rows = []
for model_name in ['dummy', 'logistic_regression']:
    for split_name in ['validation', 'test']:
        metrics = m3_report['predictive_models'][model_name][split_name]
        comparison_rows.append({
            'modelo': model_name,
            'split': split_name,
            'pr_auc': metrics['average_precision'],
            'roc_auc': metrics['roc_auc'],
            'brier': metrics['brier_score'],
            'f1': metrics['f1'],
            'recall': metrics['recall'],
            'precision': metrics['precision'],
        })
model_comparison = pd.DataFrame(comparison_rows)
model_comparison

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=model_comparison, x='split', y='pr_auc', hue='modelo', ax=axes[0])
axes[0].set(title='PR-AUC por split', xlabel='Split', ylabel='PR-AUC')
sns.barplot(data=model_comparison, x='split', y='brier', hue='modelo', ax=axes[1])
axes[1].set(title='Brier score por split — menor é melhor', xlabel='Split', ylabel='Brier')
plt.tight_layout()
plt.show()

PR-AUC é a métrica principal devido à prevalência positiva de aproximadamente 11,27%. A Regressão Logística também reduz o Brier score contra o Dummy, portanto um calibrador adicional não foi aplicado nesta baseline.

## 4. Limiar e matriz de confusão

In [ ]:
logistic_report = m3_report['predictive_models']['logistic_regression']
threshold_summary = pd.Series({
    'estrategia': 'max_f1_on_validation',
    'limiar_congelado': logistic_report['threshold_selection']['threshold'],
    'f1_validacao': logistic_report['validation']['f1'],
    'f1_teste': logistic_report['test']['f1'],
})
threshold_summary

In [ ]:
confusion = logistic_report['test']['confusion_matrix']
confusion_frame = pd.DataFrame(
    [
        [confusion['true_negatives'], confusion['false_positives']],
        [confusion['false_negatives'], confusion['true_positives']],
    ],
    index=['real_0', 'real_1'],
    columns=['predito_0', 'predito_1'],
)
sns.heatmap(confusion_frame, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de confusão — teste')
plt.show()

O limiar maximiza F1 na validação e é aplicado sem ajuste no teste. Ele é uma referência técnica; um limiar de produção deverá refletir custos de contato, capacidade e risco aprovados pelo negócio.

## 5. Calibração no teste

In [ ]:
calibration_frame = pd.DataFrame(logistic_report['test']['calibration_curve'])
figure, axis = plt.subplots(figsize=(6, 5))
axis.plot([0, 1], [0, 1], linestyle='--', color='gray', label='calibração perfeita')
axis.plot(
    calibration_frame['mean_predicted_probability'],
    calibration_frame['observed_fraction'],
    marker='o',
    label='regressão logística',
)
axis.set(
    title='Curva de calibração — teste',
    xlabel='Probabilidade média prevista',
    ylabel='Fração observada',
)
axis.legend()
plt.show()

## 6. Avaliação por fatias de auditoria

In [ ]:
slice_metrics = pd.read_csv(modeling_config.test_slice_metrics_path)
slice_metrics.sort_values(['slice_column', 'average_precision'], ascending=[True, False])

As 34 fatias com pelo menos 100 registros são avaliadas somente depois da predição. Esses campos não foram usados no treino. PR-AUC depende da prevalência de cada fatia; diferenças são sinais para investigação e não comprovam discriminação ou qualidade superior de um grupo.

## 7. Coeficientes e limites de interpretação

In [ ]:
coefficient_frame = pd.read_csv(modeling_config.coefficients_path)
pd.concat([coefficient_frame.head(8), coefficient_frame.tail(8)])

Coeficientes descrevem associações condicionais do modelo regularizado e dependem da codificação/escala. Não devem ser convertidos em regras causais ou justificativas individuais. Campos demográficos e financeiros de auditoria não foram usados.

## 8. Reuso do pipeline treinado

In [ ]:
model_artifact = joblib.load(modeling_config.model_path)
test_payload = splits['test'].loc[:, PROPENSITY_INPUT_COLUMNS].iloc[[0]]
smoke_probability = model_artifact['pipeline'].predict_proba(test_payload)[0, 1]

pd.Series({
    'fit_split': model_artifact['fit_split'],
    'selection_split': model_artifact['selection_split'],
    'limiar': model_artifact['threshold'],
    'probabilidade_smoke_test': smoke_probability,
    'duration_presente': 'duracao_contato' in model_artifact['input_columns'],
})

## 9. Tracking MLflow

In [ ]:
tracking_path = modeling_config.latest_mlflow_run_path
tracking_info = json.loads(tracking_path.read_text(encoding='utf-8')) if tracking_path.exists() else {}
pd.Series(tracking_info, dtype='object')

O run oficial registra parâmetros, métricas de validação/teste, hashes e artefatos no backend SQLite local. O notebook não cria runs adicionais para evitar poluir o histórico.

## 10. Conclusões

- `celular` é o baseline fixo, definido exclusivamente pelo treino.
- A Regressão Logística supera o Dummy em PR-AUC e Brier score na validação.
- O limiar foi escolhido na validação e congelado antes do teste.
- O pipeline serializado executa inferência sobre um payload novo com o mesmo preprocessing.
- O teste foi auditado em 34 fatias agregadas sem usar esses campos como features.
- MLflow registra parâmetros, métricas e artefatos em SQLite local.
- Os resultados permanecem observacionais e não medem o efeito causal do canal.

O M4 implementará Thompson Sampling e comparará a política adaptativa com esta regra fixa em um protocolo de replay comum.